# Nettoyage des transactions — Online Retail II

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

from src.load_data import charger_ventes_en_ligne
from src.clean_transactions import nettoyer_transactions, separer_ventes_clients
from src.validation import valider_transactions, resumer_annulations

donnees_brutes = charger_ventes_en_ligne()
donnees_brutes.shape

(1067371, 8)

## Application du pipeline de nettoyage

`nettoyer_transactions()` (voir `src/clean_transactions.py`) ajoute, dans l'ordre :
- `is_cancellation` / `transaction_type` (vente, annulation, ajustement négatif, quantité nulle)
- `description_clean` (description manquante récupérée via le StockCode, texte normalisé)
- `country_clean` (pays normalisés)
- `quantity_outlier` (au-delà du 99,9e percentile en valeur absolue)
- `line_revenue` (quantity × unit_price)

In [2]:
donnees_propres = nettoyer_transactions(donnees_brutes)
donnees_propres.head()

,invoice_no,stock_code,description,quantity,invoice_date,unit_price,customer_id,country,is_cancellation,transaction_type,description_clean,country_clean,quantity_outlier,line_revenue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,False,sale,15CM CHRISTMAS GLASS BALL 20 LIGHTS,United Kingdom,False,83.4
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,sale,PINK CHERRY LIGHTS,United Kingdom,False,81.0
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,False,sale,WHITE CHERRY LIGHTS,United Kingdom,False,81.0
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,False,sale,"RECORD FRAME 7"" SINGLE SIZE",United Kingdom,False,100.8
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,False,sale,STRAWBERRY CERAMIC TRINKET BOX,United Kingdom,False,30.0


## Répartition par type de transaction

In [3]:
donnees_propres["transaction_type"].value_counts()

transaction_type
sale                   1044420
cancellation             19494
negative_adjustment       3457
Name: count, dtype: int64

## Résumé ventes vs annulations

In [4]:
resumer_annulations(donnees_propres)

,lignes,factures,quantite,valeur
is_cancellation,,,,
False,1047877,45336,11099484,2.081392e+07
True,19494,8292,-490992,-1.526668e+06


## Suppression des doublons exacts uniquement

Les doublons potentiels (même facture/produit/quantité mais pouvant être deux saisies
légitimes) ne sont pas supprimés à ce stade — voir `docs/data_quality_report.md`.

In [5]:
avant = donnees_propres.shape[0]
donnees_propres = donnees_propres.drop_duplicates()
apres = donnees_propres.shape[0]
print(f"Lignes avant : {avant} — après : {apres} — supprimées : {avant - apres}")

Lignes avant : 1067371 — après : 1033036 — supprimées : 34335


## Séparation ventes / clients connus

In [6]:
sales_df, customer_df = separer_ventes_clients(donnees_propres)
print("sales_df :", sales_df.shape)
print("customer_df :", customer_df.shape)

sales_df : (1033036, 14)
customer_df : (797885, 14)


## Validation des règles de qualité

In [7]:
valider_transactions(donnees_propres)

,regle,respectee
0,facture_avec_date,True
1,facture_avec_pays,True
2,stock_code_present,True
3,prix_numerique,True
4,quantite_numerique,True
5,type_transaction_defini,True
6,revenu_ligne_coherent,True


## Export vers data/processed/

In [8]:
chemin_sortie = Path("../data/processed/transactions_clean.csv")
donnees_propres.to_csv(chemin_sortie, index=False)
print("Écrit :", chemin_sortie.resolve())

Écrit : C:\Users\modou\OneDrive\Documents\COURS\BacInformatiqueUQAR\Revision\DataAnalyst\Python\online-retail-customer-analytics\data\processed\transactions_clean.csv
